# ⚡ Notebook 3: Benchmark Spark — Modo Local
**Perfil:** Comunidad y profesores | **Entorno:** JupyterHub / K3s  
**Objetivo:** Medir las limitaciones computacionales de PySpark en modo `local[*]` confinado a **2 CPU / 4 GB RAM**.  
Los tiempos de este notebook serán el **baseline de comparación** contra el Notebook 4 (modo distribuido K8s).   
**Autor:** abdzher.

---
> **Modo `local[*]`:** Spark corre como un único proceso JVM dentro del pod.  
> No se crean pods adicionales. Todo el procesamiento se realiza en los 2 CPU del pod estudiante.  
> El límite de heap de la JVM estará restringido por el cgroup del pod (~3.5GB usable).

## 0. Verificación del Entorno

In [ ]:
import os, platform, psutil, subprocess, time

print("="*65)
print("ENTORNO DE EJECUCIÓN — BENCHMARK SPARK LOCAL")
print("="*65)
print(f"  Pod hostname:      {platform.node()}")
print(f"  CPUs disponibles:  {os.cpu_count()}  (límite cgroup: 2)")
mem = psutil.virtual_memory()
print(f"  RAM total visible: {mem.total/1024**3:.2f} GB (límite K8s: 4 GB)")

# Verificar Java (requerido por Spark)
try:
    java_ver = subprocess.check_output(["java", "-version"],
                                        stderr=subprocess.STDOUT).decode()
    print(f"  Java:              {java_ver.splitlines()[0]}")
except Exception as e:
    print(f"  Java:              ⚠️ No encontrado — {e}")

# JAVA_HOME para PySpark
java_home = os.environ.get("JAVA_HOME", "no definido")
print(f"  JAVA_HOME:         {java_home}")
print("="*65)

## 1. Configuración de la Sesión Spark (Modo Local)

Se configura Spark para operar **completamente dentro del pod** (sin orquestación K8s).  
El Driver y los Executors son el mismo proceso. El paralelismo máximo = número de CPUs del pod.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, FloatType, IntegerType, StringType
import time

print("🚀 Iniciando sesión Spark en modo local[*]...")
t0 = time.perf_counter()

spark = (
    SparkSession.builder
    .appName("Benchmark-Local-Estudiante")
    .master("local[*]")              # Usa todos los CPUs del pod (máx 2 por cgroup)
    .config("spark.driver.memory", "3g")  # Dejar 1GB para el SO y JupyterLab
    .config("spark.driver.maxResultSize", "512m")
    .config("spark.sql.shuffle.partitions", "8")   # Ajustado para 2 CPU
    .config("spark.ui.enabled", "false")            # Desactivar UI para ahorrar RAM
    .config("spark.eventLog.enabled", "false")
    .config("spark.local.dir", "/tmp/spark-local")
    .getOrCreate()
)

t_start = time.perf_counter() - t0
print(f"✅ Sesión Spark iniciada en {t_start:.2f}s")
print(f"   Versión Spark:    {spark.version}")
print(f"   Master:           {spark.sparkContext.master}")
print(f"   App ID:           {spark.sparkContext.applicationId}")
print(f"   App Name:         {spark.sparkContext.appName}")
print(f"   Default Parallelism: {spark.sparkContext.defaultParallelism}")
print()
print("⚠️  MODO LOCAL: Todos los cálculos corren en este único pod.")
print("   NO se crean pods Executor en Kubernetes.")

## 2. Generación del DataFrame Sintético Masivo

Generamos **30 millones de filas** simulando datos de telemetría ambiental:  
`station_id`, `timestamp_epoch`, `temperature_C`, `humidity_pct`, `pressure_hPa`, `zone`.

Usamos `spark.range()` para generación distribuida eficiente en memoria.

In [ ]:
import math

N_ROWS = 30_000_000  # 30 millones de filas
N_PARTITIONS = 8     # = shuffle.partitions, ajustado para 2 CPU

print(f"📊 Generando DataFrame sintético: {N_ROWS:,} filas × 6 columnas")
print(f"   Particiones: {N_PARTITIONS}")
print()

t0 = time.perf_counter()

# Generación usando spark.range() + expresiones de columna
# Esto es LAZY — no ejecuta nada todavía
df_spark = (
    spark.range(0, N_ROWS, numPartitions=N_PARTITIONS)
    .withColumnRenamed("id", "row_id")
    .withColumn("station_id",
                (F.col("row_id") % 500).cast(IntegerType()))          # 500 estaciones únicas
    .withColumn("timestamp_epoch",
                (1_700_000_000 + F.col("row_id")).cast("long"))       # timestamps consecutivos
    .withColumn("temperature_C",
                (F.rand(seed=42) * 60 - 10).cast(FloatType()))        # [-10°C, 50°C]
    .withColumn("humidity_pct",
                (F.rand(seed=7)  * 100).cast(FloatType()))            # [0%, 100%]
    .withColumn("pressure_hPa",
                (F.rand(seed=13) * 200 + 900).cast(FloatType()))      # [900, 1100 hPa]
    .withColumn("zone",
                F.when(F.col("station_id") < 100, F.lit("Norte"))
                 .when(F.col("station_id") < 200, F.lit("Sur"))
                 .when(F.col("station_id") < 350, F.lit("Este"))
                 .otherwise(F.lit("Oeste")))
    .drop("row_id")
)

print(f"✅ Plan lógico creado (lazy) en {(time.perf_counter()-t0)*1000:.1f} ms")
print(f"   Schema:")
df_spark.printSchema()
print("   Nota: Los datos NO están en memoria todavía (evaluación perezosa).")

## 3. Benchmark — Conteo Total (Acción 1)

El `.count()` fuerza la **materialización completa** del DataFrame.  
Es la acción más cara y nos da el tiempo de referencia base.

In [ ]:
print("⏱️  BENCHMARK 1: count() — Materialización completa del DataFrame")
print("-"*60)
print(f"   Filas objetivo: {N_ROWS:,}")
print()

t0 = time.perf_counter()
count_result = df_spark.count()
t_count = time.perf_counter() - t0

print(f"   ✅ Filas contadas:  {count_result:,}")
print(f"   ⏱️  Tiempo:         {t_count:.3f} s")
print(f"   📊 Throughput:     {count_result/t_count/1e6:.2f} M filas/segundo")

TIEMPO_COUNT_LOCAL = t_count  # Guardar para comparación con NB4

## 4. Benchmark — Agregaciones Grupales (Acción 2)

GroupBy por `zone` y `station_id` calculando estadísticas: media, mín, máx, desvío estándar.  
Esta operación involucra un **shuffle** completo de los datos entre particiones.

In [ ]:
print("⏱️  BENCHMARK 2: GroupBy con Agregaciones (con shuffle)")
print("-"*60)

df_agg = (
    df_spark
    .groupBy("zone", "station_id")
    .agg(
        F.count("*").alias("num_mediciones"),
        F.avg("temperature_C").alias("temp_media_C"),
        F.min("temperature_C").alias("temp_min_C"),
        F.max("temperature_C").alias("temp_max_C"),
        F.stddev("temperature_C").alias("temp_stddev_C"),
        F.avg("humidity_pct").alias("hum_media_pct"),
        F.avg("pressure_hPa").alias("pres_media_hPa"),
    )
    .orderBy("zone", "station_id")
)

t0 = time.perf_counter()
result_agg = df_agg.collect()  # Acción que materializa el resultado
t_agg = time.perf_counter() - t0

print(f"   ✅ Grupos generados: {len(result_agg):,}")
print(f"   ⏱️  Tiempo:         {t_agg:.3f} s")
print()
print("   Muestra de resultados (primeros 5):")
for row in result_agg[:5]:
    print(f"     Zona={row.zone:5s} | Estación={row.station_id:3d} "
          f"| Mediciones={row.num_mediciones:,} "
          f"| Temp_Media={row.temp_media_C:.2f}°C")

TIEMPO_AGG_LOCAL = t_agg

## 5. Benchmark — Join Complejo (Acción 3)

Simulamos un **join** entre el dataset principal y una tabla de referencia de estaciones  
(enriquecimiento de datos). Esta operación es costosa porque requiere redistribuir datos  
por la clave de join entre todas las particiones.

In [ ]:
print("⏱️  BENCHMARK 3: Join DataFrame principal × Tabla de referencia")
print("-"*60)

# Tabla de referencia pequeña (500 estaciones con metadatos)
import pandas as pd
import numpy as np

np.random.seed(99)
stations_pd = pd.DataFrame({
    "station_id":   range(500),
    "lat":          np.random.uniform(-40, 40, 500).round(4),
    "lon":          np.random.uniform(-120, 120, 500).round(4),
    "elevation_m":  np.random.randint(0, 4500, 500),
    "sensor_type":  np.random.choice(["Type-A", "Type-B", "Type-C"], 500),
})

# Broadcast join: la tabla pequeña se envía a todos los executors
stations_spark = spark.createDataFrame(stations_pd)
stations_broadcast = F.broadcast(stations_spark)

df_joined = (
    df_spark
    .join(stations_broadcast, on="station_id", how="left")
    .select(
        "zone", "station_id", "sensor_type", "elevation_m",
        "temperature_C", "humidity_pct", "pressure_hPa"
    )
)

# Filtrar y agregar DESPUÉS del join
df_final = (
    df_joined
    .filter(F.col("elevation_m") > 2000)  # Solo estaciones de alta montaña
    .groupBy("zone", "sensor_type")
    .agg(
        F.count("*").alias("registros"),
        F.avg("temperature_C").alias("temp_media"),
        F.avg("pressure_hPa").alias("presion_media"),
    )
)

t0 = time.perf_counter()
result_join = df_final.collect()
t_join = time.perf_counter() - t0

print(f"   ✅ Filas resultado del join: {len(result_join)}")
print(f"   ⏱️  Tiempo:                 {t_join:.3f} s")
print()
print("   Resultados (Estaciones alta montaña):")
for row in sorted(result_join, key=lambda r: r.zone):
    print(f"     Zona={row.zone:5s} | Sensor={row.sensor_type} "
          f"| Registros={row.registros:,} "
          f"| Temp={row.temp_media:.2f}°C")

TIEMPO_JOIN_LOCAL = t_join

## 6. Resumen de Tiempos — Modo Local (Baseline)

Estos valores son el **baseline** para comparar contra el modo distribuido K8s del Notebook 4.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

benchmark_results = {
    "count()\n(30M filas)": TIEMPO_COUNT_LOCAL,
    "GroupBy +\nAgregaciones": TIEMPO_AGG_LOCAL,
    "Broadcast Join\n+ Filter + GroupBy": TIEMPO_JOIN_LOCAL,
}

print("="*65)
print("  RESUMEN DE BENCHMARKS — SPARK LOCAL (2 CPU / 4GB RAM)")
print("="*65)
total = 0
for op, t in benchmark_results.items():
    label = op.replace("\n", " ")
    print(f"  {label:<40} {t:>8.3f} s")
    total += t
print("-"*65)
print(f"  {'TOTAL':<40} {total:>8.3f} s")
print("="*65)
print()
print("  📌 Guardar estos valores para comparar con NB4 (modo K8s distribuido).")
print(f"     TIEMPO_COUNT_LOCAL = {TIEMPO_COUNT_LOCAL:.3f} s")
print(f"     TIEMPO_AGG_LOCAL   = {TIEMPO_AGG_LOCAL:.3f} s")
print(f"     TIEMPO_JOIN_LOCAL  = {TIEMPO_JOIN_LOCAL:.3f} s")

# Gráfico de barras
fig, ax = plt.subplots(figsize=(10, 5))
labels = list(benchmark_results.keys())
values = list(benchmark_results.values())
bars = ax.bar(labels, values, color=["#e74c3c", "#e67e22", "#c0392b"],
              edgecolor="white", linewidth=0.8, width=0.5)

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{val:.1f}s", ha="center", va="bottom", fontsize=11, fontweight="bold")

ax.set_ylabel("Tiempo (segundos)", fontsize=11)
ax.set_title("Benchmark Spark LOCAL — Perfil Estudiante (2 CPU / 4GB)\n"
             "Baseline para comparación con modo K8s distribuido",
             fontsize=11, fontweight="bold")
ax.set_ylim(0, max(values) * 1.25)
local_patch = mpatches.Patch(color='#e74c3c', label='Modo local[*] (pod único)')
ax.legend(handles=[local_patch], fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig("/tmp/nb3_benchmark_local.png", dpi=120, bbox_inches="tight")
plt.show()
print("✅ Benchmark guardado.")

In [ ]:
# Cerrar la sesión Spark para liberar recursos
spark.stop()
print("✅ Sesión Spark detenida. Recursos liberados.")

---
## Resumen del Notebook 3

| Operación | Tiempo (modo local) | Observación |
|---|---|---|
| `count()` 30M filas | ~X s | CPU-bound, cuello de botella en 2 CPUs |
| GroupBy + Agregaciones | ~X s | Shuffle costoso con pocas particiones |
| Broadcast Join + GroupBy | ~X s | Mejor por broadcast, aún limitado |

> **Conclusión:** En modo `local[*]` con 2 CPU, el procesamiento de 30M filas tarda varios minutos.  
> El **Notebook 4** demostrará cómo el modo K8s con 3 executors reduce estos tiempos significativamente.